In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

**# **Cell 0 — Concat RRF parts****

In [ ]:
from pathlib import Path
import shutil

UMLS_BASE = Path('/kaggle/input/datasets/konicarokeya/umls-2025ab-parts')
UMLS_OUT  = Path('/kaggle/working/umls')
UMLS_OUT.mkdir(exist_ok=True)

def concat_parts(part_names, output_name):
    out_path = UMLS_OUT / output_name
    if out_path.exists():
        print(f'{output_name} already exists ({out_path.stat().st_size/1024**2:.0f} MB) — skipping')
        return out_path
    print(f'Building {output_name} from {len(part_names)} parts...')
    with open(out_path, 'wb') as out_f:
        for name in sorted(part_names):
            p = UMLS_BASE / name / name
            print(f'  + {name}  ({p.stat().st_size/1024**2:.0f} MB)')
            with open(p, 'rb') as in_f:
                shutil.copyfileobj(in_f, out_f)
    print(f'  done -> {out_path.stat().st_size/1024**2:.0f} MB total\n')
    return out_path

MRCONSO_PATH = concat_parts(
    ['MRCONSO.RRF.aa', 'MRCONSO.RRF.ab', 'MRCONSO.RRF.ac'],
    'MRCONSO.RRF'
)
MRREL_PATH = concat_parts(
    ['MRREL.RRF.aa', 'MRREL.RRF.ab', 'MRREL.RRF.ac',
     'MRREL.RRF.ad', 'MRREL.RRF.ae'],
    'MRREL.RRF'
)
MRSTY_PATH  = UMLS_BASE / 'MRSTY.RRF' / 'MRSTY.RRF'
CORPUS_PATH = Path('/kaggle/input/datasets/konicarokeya/mesh-complete/retrieval_corpus_MESH_COMPLETE.parquet')

print('='*50)
print(f'MRCONSO : {MRCONSO_PATH}')
print(f'MRREL   : {MRREL_PATH}')
print(f'MRSTY   : {MRSTY_PATH}  exists={MRSTY_PATH.exists()}')
print(f'CORPUS  : {CORPUS_PATH}  exists={CORPUS_PATH.exists()}')

**Cell 1 — Imports**

In [ ]:
import gc
import math
import pickle
import shutil
from pathlib import Path
from collections import defaultdict
from itertools import combinations
from tqdm.auto import tqdm
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

print('Imports OK')

**Cell 2 — Config**

In [ ]:
HKG_OUT = Path('/kaggle/working/hkg.pkl')
TARGET_SABS = {'MSH', 'SNOMEDCT_US'}
MIN_COOCCUR = 2

ALLOWED_TUIS = {
    # Disorders & Disease
    'T047','T048','T049','T050','T191','T046','T184',
    # Chemicals & Drugs
    'T121','T109','T195','T200','T116','T126','T127',
    'T131','T125','T129','T130','T123','T122','T197',
    'T103','T120',
    # Anatomy
    'T023','T024','T025','T026','T029','T030','T031','T022',
    # Physiology
    'T039','T040','T041','T042','T043','T044','T045','T038',
    # Procedures
    'T061','T058','T059','T060','T065','T063','T062',
    # Population & Demographics
    'T016','T096','T098','T099','T032','T100','T080',
    # Animals & Organisms
    'T008','T015','T007','T004','T005','T001','T002',
    'T011','T012','T013','T014',
    # Genes & Molecular Biology
    'T028','T086','T114','T087','T085',
    # Findings & Results
    'T033','T034','T201','T190',
    # Research & Concepts
    'T081','T077','T169','T090','T052','T079','T078',
    'T170','T171',
    # Devices
    'T074','T075','T073',
    # Clinical
    'T037','T020','T019','T018','T017',
    # Diet & Nutrition
    'T168',
    # Lifestyle & Behaviors
    'T053','T054','T055',
    # Environmental
    'T069','T070',
}

print(f'Config OK — {len(ALLOWED_TUIS)} semantic types')

**Cell 3 — Load corpus**

In [ ]:
print('Loading corpus...')
df = pd.read_parquet(CORPUS_PATH)
print(f'Rows    : {len(df):,}')
print(f'Sources : {df["source"].value_counts().to_dict()}')

def normalize_mesh(m):
    if m is None: return []
    if isinstance(m, np.ndarray): m = m.tolist()
    if not isinstance(m, list): return []
    out = []
    for x in m:
        if isinstance(x, dict):
            term   = x.get('term', '')
            meshid = x.get('mesh_id', '')
            if term and str(term).strip():
                out.append({
                    'term'   : str(term).strip().lower(),
                    'mesh_id': str(meshid).strip() if meshid else ''
                })
    return out

df['meshes_norm'] = df['meshes'].apply(normalize_mesh)

term_to_chunks   = defaultdict(list)
meshid_to_chunks = defaultdict(list)

for idx, mesh_list in enumerate(tqdm(df['meshes_norm'], desc='Building chunk index')):
    for entry in mesh_list:
        t = entry['term']
        m = entry['mesh_id']
        if t:
            term_to_chunks[t].append(idx)
        if m and m.startswith('D'):
            meshid_to_chunks[m].append(idx)

corpus_texts = df['text'].tolist()

print(f'\nUnique terms    : {len(term_to_chunks):,}')
print(f'Unique mesh_ids : {len(meshid_to_chunks):,}')
print(f'Corpus size     : {len(corpus_texts):,}')
gc.collect()

**Cell 4 — Load MRSTY**

In [ ]:
print('Loading MRSTY...')

allowed_cuis = set()
cui_to_tui   = {}

with open(MRSTY_PATH, encoding='utf-8') as f:
    for line in tqdm(f, desc='MRSTY'):
        row = line.rstrip('\n').split('|')
        if len(row) < 2: continue
        cui = row[0]
        tui = row[1]
        cui_to_tui[cui] = tui
        if tui in ALLOWED_TUIS:
            allowed_cuis.add(cui)

print(f'Total CUIs       : {len(cui_to_tui):,}')
print(f'Allowed CUIs     : {len(allowed_cuis):,}')
gc.collect()

**Cell 5 — Load MRCONSO (fixed ispref bug)**

In [ ]:
print('Loading MRCONSO...')

G             = nx.DiGraph()
label_to_cui  = {}
meshid_to_cui = {}

with open(MRCONSO_PATH, encoding='utf-8') as f:
    for line in tqdm(f, desc='MRCONSO'):
        row = line.rstrip('\n').split('|')
        if len(row) < 15: continue
        cui    = row[0]
        lang   = row[1]
        ispref = row[6]
        sab    = row[11]
        code   = row[13]
        name   = row[14]

        if lang != 'ENG': continue
        if sab not in TARGET_SABS: continue
        if cui not in allowed_cuis: continue

        label = name.lower().strip()

        # Add node if not exists
        if cui not in G:
            G.add_node(cui, label=label, sab=sab, chunk_idxs=[])

        # Only update display label with preferred term
        if ispref == 'Y':
            G.nodes[cui]['label'] = label

        # Map ALL terms and IDs regardless of ispref — fixes missing nodes
        if label:
            label_to_cui[label] = cui

        if sab == 'MSH' and code.startswith('D'):
            meshid_to_cui[code] = cui

print(f'Nodes loaded  : {G.number_of_nodes():,}')
print(f'label_to_cui  : {len(label_to_cui):,}')
print(f'meshid_to_cui : {len(meshid_to_cui):,}')
gc.collect()

**Cell 6 — Load MRREL**

In [ ]:
print('Loading MRREL...')

VALID_RELS = {'PAR','CHD','RB','RN','RO','SY'}
edge_count = 0

with open(MRREL_PATH, encoding='utf-8') as f:
    for line in tqdm(f, desc='MRREL'):
        row = line.rstrip('\n').split('|')
        if len(row) < 8: continue
        cui1 = row[0]
        rel  = row[3]
        cui2 = row[4]
        rela = row[7]

        if rel not in VALID_RELS: continue
        if cui1 not in G: continue
        if cui2 not in G: continue

        G.add_edge(cui1, cui2, rel=rel, rela=rela or rel)
        edge_count += 1

print(f'UMLS edges added : {edge_count:,}')
print(f'Total edges now  : {G.number_of_edges():,}')
gc.collect()

**Cell 7 — Connect corpus chunks**

In [ ]:
print('Connecting corpus chunks to graph nodes...')

connected_via_meshid = 0
connected_via_string = 0
no_node_found        = 0

for idx, mesh_list in enumerate(tqdm(df['meshes_norm'], desc='Connecting')):
    for entry in mesh_list:
        term   = entry['term']
        meshid = entry['mesh_id']

        if meshid and meshid in meshid_to_cui:
            cui = meshid_to_cui[meshid]
            if cui in G:
                G.nodes[cui]['chunk_idxs'].append(idx)
                connected_via_meshid += 1
                continue

        if term and term in label_to_cui:
            cui = label_to_cui[term]
            if cui in G:
                G.nodes[cui]['chunk_idxs'].append(idx)
                connected_via_string += 1
                continue

        no_node_found += 1

# Deduplicate
for node in G.nodes():
    G.nodes[node]['chunk_idxs'] = list(set(G.nodes[node]['chunk_idxs']))

reachable = sum(1 for n in G.nodes() if G.nodes[n]['chunk_idxs'])
print(f'\nConnected via mesh_id : {connected_via_meshid:,}')
print(f'Connected via string  : {connected_via_string:,}')
print(f'No node found         : {no_node_found:,}')
print(f'Nodes with chunks     : {reachable:,} / {G.number_of_nodes():,}')
gc.collect()

In [ ]:
test_terms = ['diabetes mellitus', 'metformin', 'hypertension',
              'female', 'aged', 'rats', 'treatment outcome',
              'breast neoplasms', 'insulin', 'inflammation']

print('Traversal check:')
for term in test_terms:
    cui = label_to_cui.get(term)
    if not cui:
        print(f'  {term:30s} → NOT FOUND')
        continue
    chunks = G.nodes[cui].get('chunk_idxs', [])
    neighbors = [G.nodes[n].get('label','?') 
                 for n in list(G.successors(cui))[:3]]
    print(f'  {term:30s} → {len(chunks):,} chunks | {neighbors}')

In [ ]:
# Check exactly what is happening for metformin and rats
check_terms = ['metformin', 'rats', 'breast neoplasms', 
               'insulin', 'inflammation', 'treatment outcome']

for term in check_terms:
    print(f'\n--- {term} ---')
    
    # What label_to_cui gives
    cui_from_label = label_to_cui.get(term)
    print(f'  label_to_cui CUI      : {cui_from_label}')
    if cui_from_label:
        chunks = G.nodes[cui_from_label].get('chunk_idxs', [])
        print(f'  chunks on that CUI    : {len(chunks)}')
    
    # What mesh_id gives
    if term in term_to_chunks:
        print(f'  term_to_chunks count  : {len(term_to_chunks[term])}')
    
    # Check D-number
    matching_d = [mid for mid, chunks in meshid_to_chunks.items() 
                  if term in mid.lower()]
    print(f'  matching D-numbers    : {matching_d[:3]}')
    
    # Check which CUI has the actual chunks
    for mid, idx_list in list(meshid_to_chunks.items())[:]:
        if len(idx_list) > 0:
            # Check if any chunk text mentions the term
            sample_idx = idx_list[0]
            if term in corpus_texts[sample_idx].lower():
                cui = meshid_to_cui.get(mid)
                print(f'  found via D-number    : {mid} → CUI={cui} → {len(idx_list)} chunks')
                break

In [ ]:
print('Connecting corpus chunks to graph nodes...')

connected_via_meshid = 0
connected_via_string = 0
connected_via_term   = 0
no_node_found        = 0

for idx, mesh_list in enumerate(tqdm(df['meshes_norm'], desc='Connecting')):
    for entry in mesh_list:
        term   = entry['term']
        meshid = entry['mesh_id']

        # Method 1: D-number → CUI (most reliable)
        if meshid and meshid in meshid_to_cui:
            cui = meshid_to_cui[meshid]
            if cui in G:
                G.nodes[cui]['chunk_idxs'].append(idx)
                connected_via_meshid += 1
                continue

        # Method 2: term string → CUI via label_to_cui
        if term and term in label_to_cui:
            cui = label_to_cui[term]
            if cui in G:
                G.nodes[cui]['chunk_idxs'].append(idx)
                connected_via_string += 1
                continue

        no_node_found += 1

# ── NEW: also attach via term_to_chunks directly ──────────────
# For terms that exist in corpus but their CUI got no chunks
# above, use term_to_chunks to find all chunk indices and
# attach them to whatever CUI that term maps to
print('\nApplying term_to_chunks fallback...')
fallback_count = 0
for term, chunk_idxs in tqdm(term_to_chunks.items(),
                              desc='Term fallback'):
    cui = label_to_cui.get(term)
    if not cui or cui not in G:
        continue
    existing = set(G.nodes[cui]['chunk_idxs'])
    new_idxs = [i for i in chunk_idxs if i not in existing]
    if new_idxs:
        G.nodes[cui]['chunk_idxs'].extend(new_idxs)
        fallback_count += len(new_idxs)

# Deduplicate all
for node in G.nodes():
    G.nodes[node]['chunk_idxs'] = list(set(G.nodes[node]['chunk_idxs']))

reachable = sum(1 for n in G.nodes() if G.nodes[n]['chunk_idxs'])
print(f'\nConnected via mesh_id : {connected_via_meshid:,}')
print(f'Connected via string  : {connected_via_string:,}')
print(f'Fallback via term     : {fallback_count:,}')
print(f'No node found         : {no_node_found:,}')
print(f'Nodes with chunks     : {reachable:,} / {G.number_of_nodes():,}')
gc.collect()

In [ ]:
test_terms = ['diabetes mellitus', 'metformin', 'hypertension',
              'female', 'aged', 'rats', 'treatment outcome',
              'breast neoplasms', 'insulin', 'inflammation']

print('Traversal check:')
for term in test_terms:
    cui = label_to_cui.get(term)
    if not cui:
        print(f'  {term:30s} → NOT FOUND')
        continue
    chunks    = G.nodes[cui].get('chunk_idxs', [])
    neighbors = [G.nodes[n].get('label','?')
                 for n in list(G.successors(cui))[:3]]
    print(f'  {term:30s} → {len(chunks):,} chunks | {neighbors}')

**Cell 8 — Co-occurrence edges (fixed hash bug)**

In [ ]:
print('Building co-occurrence edges...')

cooccur = defaultdict(int)

for idx, mesh_list in enumerate(tqdm(df['meshes_norm'], desc='Co-occurrence')):
    terms = list(set(
        meshid_to_cui.get(e['mesh_id'], label_to_cui.get(e['term']))
        for e in mesh_list
        if meshid_to_cui.get(e['mesh_id']) or label_to_cui.get(e['term'])
    ))
    terms = [t for t in terms if t and t in G]

    # Fixed: sort tuple so (A,B) and (B,A) counted as same edge
    for a, b in combinations(terms, 2):
        edge = tuple(sorted([a, b]))
        cooccur[edge] += 1

co_edges = 0
for (a, b), w in cooccur.items():
    if w >= MIN_COOCCUR:
        G.add_edge(a, b, rel='CO_OCCUR', weight=w)
        G.add_edge(b, a, rel='CO_OCCUR', weight=w)
        co_edges += 1

print(f'Co-occurrence edges added : {co_edges:,}')
print(f'Total edges now           : {G.number_of_edges():,}')
del cooccur
gc.collect()

**Cell 9 — Save**

In [ ]:
print('Saving HKG...')

payload = {
    'graph'           : G,
    'term_to_chunks'  : dict(term_to_chunks),
    'meshid_to_chunks': dict(meshid_to_chunks),
    'label_to_cui'    : label_to_cui,
    'meshid_to_cui'   : meshid_to_cui,
}

with open(HKG_OUT, 'wb') as f:
    pickle.dump(payload, f, protocol=4)

size_mb = HKG_OUT.stat().st_size / 1024**2
print(f'\nSaved : {HKG_OUT}  ({size_mb:.0f} MB)')
print(f'Nodes : {G.number_of_nodes():,}')
print(f'Edges : {G.number_of_edges():,}')
print('\nKG builder complete.')

**Cell 10 — Sample rows (10 nodes with details**

In [ ]:
print('='*65)
print('SAMPLE KG NODES — 10 nodes with chunks and neighbors')
print('='*65)

# Pick 10 nodes that have chunk connections
sample_nodes = [
    n for n in G.nodes()
    if G.nodes[n]['chunk_idxs']
][:10]

rows = []
for node in sample_nodes:
    data       = G.nodes[node]
    label      = data.get('label', '')
    sab        = data.get('sab', '')
    n_chunks   = len(data.get('chunk_idxs', []))
    successors = list(G.successors(node))
    n_neighbors = len(successors)
    neighbor_labels = [
        G.nodes[n].get('label', '?')
        for n in successors[:3]
    ]
    tui = cui_to_tui.get(node, '?')
    rows.append({
        'CUI'         : node,
        'Label'       : label[:30],
        'SAB'         : sab,
        'TUI'         : tui,
        'Chunks'      : n_chunks,
        'Neighbors'   : n_neighbors,
        'Top 3 neighbors' : ' | '.join(neighbor_labels[:3]),
    })

df_sample = pd.DataFrame(rows)
pd.set_option('display.max_colwidth', 40)
pd.set_option('display.width', 120)
print(df_sample.to_string(index=False))

**Cell 11 — Traversal test on real medical terms**

In [ ]:
print('='*65)
print('GRAPH TRAVERSAL TEST — real terms from your corpus')
print('='*65)

test_terms = [
    'diabetes mellitus', 'metformin', 'hypertension',
    'breast neoplasms', 'insulin', 'myocardial infarction',
    'female', 'aged', 'rats', 'treatment outcome'
]

for term in test_terms:
    cui = label_to_cui.get(term)
    if not cui:
        print(f'  {term:30s} → NOT IN GRAPH')
        continue
    chunks    = G.nodes[cui].get('chunk_idxs', [])
    neighbors = list(G.successors(cui))
    nb_labels = [G.nodes[n].get('label','?') for n in neighbors[:3]]
    tui       = cui_to_tui.get(cui, '?')
    print(f'  {term:30s} | CUI={cui} | TUI={tui} '
          f'| chunks={len(chunks):,} | neighbors={nb_labels}')

**Cell 12 — Visualizations**

In [ ]:
print('Building visualizations...')

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Hybrid Knowledge Graph — Build Summary', 
             fontsize=16, fontweight='bold')

# ── Plot 1: Node count by semantic type ──────────────────────
ax1 = axes[0, 0]
tui_labels = {
    'T047':'Disease','T121':'Drug','T023':'Anatomy',
    'T039':'Physiology','T061':'Procedure','T016':'Human',
    'T008':'Animal','T028':'Gene','T033':'Finding',
    'T062':'Research','T168':'Food','T053':'Behavior',
}
tui_counts = defaultdict(int)
for node in G.nodes():
    tui = cui_to_tui.get(node, 'Other')
    label = tui_labels.get(tui, tui)
    tui_counts[label] += 1

top_tuis = sorted(tui_counts.items(), key=lambda x: x[1], reverse=True)[:12]
labels_t, counts_t = zip(*top_tuis)
colors = plt.cm.Set3(np.linspace(0, 1, len(labels_t)))
bars = ax1.barh(labels_t, counts_t, color=colors)
ax1.set_xlabel('Node count')
ax1.set_title('Nodes by semantic type')
for bar, count in zip(bars, counts_t):
    ax1.text(bar.get_width() + 100, bar.get_y() + bar.get_height()/2,
             f'{count:,}', va='center', fontsize=8)
ax1.invert_yaxis()

# ── Plot 2: Edge type distribution ───────────────────────────
ax2 = axes[0, 1]
edge_type_counts = defaultdict(int)
for u, v, data in G.edges(data=True):
    edge_type_counts[data.get('rel', 'UNK')] += 1

et_labels = list(edge_type_counts.keys())
et_values = list(edge_type_counts.values())
colors2   = plt.cm.Pastel1(np.linspace(0, 1, len(et_labels)))
wedges, texts, autotexts = ax2.pie(
    et_values, labels=et_labels, autopct='%1.1f%%',
    colors=colors2, startangle=90
)
ax2.set_title('Edge type distribution')

# ── Plot 3: Chunks per node distribution ─────────────────────
ax3 = axes[1, 0]
chunk_counts = [
    len(G.nodes[n]['chunk_idxs'])
    for n in G.nodes()
    if G.nodes[n]['chunk_idxs']
]
ax3.hist(chunk_counts, bins=50, color='steelblue',
         edgecolor='white', alpha=0.8)
ax3.set_xlabel('Chunks per node')
ax3.set_ylabel('Number of nodes')
ax3.set_title('Chunk coverage distribution')
ax3.set_yscale('log')
ax3.axvline(np.median(chunk_counts), color='red',
            linestyle='--', label=f'Median: {np.median(chunk_counts):.0f}')
ax3.legend()

# ── Plot 4: Source coverage ───────────────────────────────────
ax4 = axes[1, 1]
source_names = ['pqaa', 'pqau', 'medrag_pubmed', 'medrag_wikipedia']
source_colors = ['#4CAF50','#2196F3','#FF9800','#9C27B0']
source_counts = []
for src in source_names:
    sub = df[df['source'] == src]
    has_mesh = sub['meshes_norm'].apply(lambda x: len(x) > 0).sum()
    source_counts.append(has_mesh)

bars2 = ax4.bar(source_names, source_counts,
                color=source_colors, alpha=0.85, edgecolor='white')
ax4.set_ylabel('Chunks with MeSH')
ax4.set_title('MeSH coverage by source')
ax4.tick_params(axis='x', rotation=15)
for bar, count in zip(bars2, source_counts):
    ax4.text(bar.get_x() + bar.get_width()/2,
             bar.get_height() + 1000,
             f'{count:,}', ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('/kaggle/working/hkg_summary.png', dpi=150,
            bbox_inches='tight')
plt.show()
print('Saved: hkg_summary.png')

**Cell 13 — Final summary stats**

In [ ]:
print('='*65)
print('FINAL KG SUMMARY')
print('='*65)

total_nodes     = G.number_of_nodes()
total_edges     = G.number_of_edges()
nodes_with_data = sum(1 for n in G.nodes() if G.nodes[n]['chunk_idxs'])
umls_edges      = sum(1 for u,v,d in G.edges(data=True)
                      if d.get('rel') != 'CO_OCCUR')
cooccur_edges   = sum(1 for u,v,d in G.edges(data=True)
                      if d.get('rel') == 'CO_OCCUR')
avg_degree      = total_edges / max(total_nodes, 1)

print(f'  Total nodes              : {total_nodes:,}')
print(f'  Nodes connected to corpus: {nodes_with_data:,} '
      f'({100*nodes_with_data/total_nodes:.1f}%)')
print(f'  Total edges              : {total_edges:,}')
print(f'  UMLS ontology edges      : {umls_edges:,}')
print(f'  Co-occurrence edges      : {cooccur_edges:,}')
print(f'  Avg degree per node      : {avg_degree:.1f}')
print(f'  HKG file size            : '
      f'{HKG_OUT.stat().st_size/1024**2:.0f} MB')
print()
print('Next step: upload hkg.pkl as Kaggle dataset hkg-graph-cache')
print('Then start Notebook B: hkg-rag-fold0_2_4_6_8-evaluation.ipynb')